importing necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "iframe"
import plotly.express as px
from textblob import TextBlob

In [ ]:
df = pd.read_csv("/Users/abhimanyuchettiar/Downloads/flipkart_com-ecommerce_sample.csv")

overview of the dataset

In [ ]:
df.head()

In [ ]:
df.info()

data cleaning

In [ ]:
#convert prices to numeric
df["retail_price"] = pd.to_numeric(df["retail_price"], errors="coerce")
df["discounted_price"] = pd.to_numeric(df["discounted_price"], errors="coerce")
#convert ratings to numeric
df["retail_price"] = pd.to_numeric(df["retail_price"], errors="coerce")
df["discounted_price"] = pd.to_numeric(df["discounted_price"], errors="coerce")

In [ ]:
#extract main product category
df["main_category"] = df["product_category_tree"].str.split(">>").str[0]
df["main_category"] = df["main_category"].str.replace('[\\["]', '', regex=True)

In [ ]:
# create discount percentage column
df["discount_percent"] = (
    (df["retail_price"] - df["discounted_price"]) / df["retail_price"]
) * 100

In [ ]:
# remove rows with missing prices
df = df.dropna(subset=["retail_price", "discounted_price"])
#clean rating column
df["product_rating"] = pd.to_numeric(df["product_rating"], errors="coerce")
ratings_df = df.dropna(subset=["product_rating"])

Data Analysis

Distribution of retail prices

In [ ]:
price_df = df[df["retail_price"] < 50000]


fig = px.histogram(
    price_df,
    x="retail_price",
    nbins=50,
    title="Distribution of Product Retail Prices"
)

fig.show()

retail price vs discounted price

In [ ]:
fig = px.scatter(
    df,
    x="retail_price",
    y="discounted_price",
    title="Retail Price vs Discounted Price",
    opacity=0.6
)

fig.show()

Top 10 brands

In [ ]:
top_brands = df["brand"].value_counts().head(10).reset_index()
top_brands.columns = ["brand", "count"]

fig = px.bar(
    top_brands,
    x="brand",
    y="count",
    title="Top 10 Most Common Brands"
)

fig.show()

Product Rating Distribution

In [ ]:


fig = px.histogram(
    df,
    x="product_rating",
    nbins=20,
    title="Distribution of Product Ratings"
)

fig.show()

Top product categories

In [ ]:
top_categories = df["main_category"].value_counts().head(10).reset_index()
top_categories.columns = ["category", "count"]

fig = px.bar(
    top_categories,
    x="category",
    y="count",
    title="Top 10 Product Categories"
)

fig.show()

Average discount by category

In [ ]:
df["product_category_tree"].head()

In [ ]:
df["main_category"] = df["product_category_tree"].str.extract(r'\["([^>]*)')
df["main_category"] = df["main_category"].str.strip()

In [ ]:
df["main_category"].value_counts().head(10)

In [ ]:
avg_discount = (
    df.groupby("main_category")["discount_percent"]
    .mean()
    .reset_index()
)

In [ ]:
category_counts = df["main_category"].value_counts()

valid_categories = category_counts[category_counts > 50].index

filtered_df = df[df["main_category"].isin(valid_categories)]

In [ ]:
avg_discount = (
    filtered_df.groupby("main_category")["discount_percent"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

In [ ]:
import plotly.express as px

fig = px.bar(
    avg_discount,
    x="main_category",
    y="discount_percent",
    title="Average Discount by Category"
)

fig.show()

Implementing machine learning models 

feature engineering

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

In [ ]:
le_category = LabelEncoder()
le_brand = LabelEncoder()

df["category_encoded"] = le_category.fit_transform(df["main_category"].astype(str))
df["brand_encoded"] = le_brand.fit_transform(df["brand"].astype(str))

In [ ]:
X = df[["discount_percent", "category_encoded", "brand_encoded"]]
y = df["retail_price"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print("Linear Regression R2:", r2_score(y_test, y_pred_lr))
print("Linear Regression RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))

In [ ]:
fig = px.scatter(
    x=y_test,
    y=y_pred_lr,
    labels={"x": "Actual Price", "y": "Predicted Price"},
    title="Linear Regression: Actual vs Predicted"
)

fig.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=5)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
print("Decision Tree R2:", r2_score(y_test, y_pred_dt))
print("Decision Tree RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_dt)))
fig = px.scatter(
    x=y_test,
    y=y_pred_dt,
    labels={"x": "Actual Price", "y": "Predicted Price"},
    title="Decision Tree: Actual vs Predicted"
)

fig.show()

In [ ]:
df["high_rating"] = (df["product_rating"] >= 4).astype(int)

df_class = df.dropna(subset=["product_rating"])

X_cls = df_class[["discount_percent", "category_encoded", "brand_encoded"]]
y_cls = df_class["high_rating"]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_c, y_train_c)

y_pred_cls = log_reg.predict(X_test_c)
print("Logistic Regression Accuracy:", accuracy_score(y_test_c, y_pred_cls))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_c, y_pred_cls)

fig = px.imshow(
    cm,
    text_auto=True,
    labels=dict(x="Predicted", y="Actual"),
    title="Confusion Matrix"
)

fig.show()